# Analysis: does manager sentiment predict market returns?

The Jiang, Lee, Martin and Zhou design, run on the full-sample manager-sentiment
series in `time_aggregation/`. The example-filing series in
`manager_sentiment_results/` is the same construction on the `input_data` sample.
Covers growth and financial risk
in one pass, since the specification is the same.

1. descriptives and comparison with the published Manager Sentiment Index
2. predictive regressions of cumulative excess market returns on sentiment,
   horizons 1 to 36 months, with and without the Goyal-Welch controls
3. the same over the original 2003-2014 sample period
4. uncertainty interactions: VIX, NVIX, recessions, and growth x financial risk

**Horizons.** `net_cum_return{h}` at month *t* compounds the excess market return
over months *t* through *t+h*, which is the earlier project's convention: the
window spans h+1 months and starts with the predictor's own month, so
`net_cum_return1` is a two-month return covering *t* and *t+1*. Pass
`skip_months=1` to `build_excess_market_returns` for the forward-only variant
(exactly h months, starting at *t+1*); the last section shows both side by side.

**Newey-West lags** scale with the horizon. Consecutive long-horizon returns
overlap almost completely, so a fixed lag length understates the standard errors
at long horizons; `maxlags="horizon"` uses *h* lags for horizon *h*.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1]  # code/analysis -> code -> repo root
sys.path.insert(0, str(REPO_ROOT / "code" / "scripts"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import econometrics as ec
import macro_data as md
import time_aggregation as ta
from prompts import STRATEGIES
from utils import INPUT_DATA_DIR, TIME_AGGREGATION_DIR, set_seed, zscore

pd.set_option("display.max_columns", None)
set_seed(42)

MONTH = ta.MONTH_COL
HORIZONS = md.DEFAULT_HORIZONS
print("horizons:", HORIZONS)

## Inputs

Market data files are expected in `input_data/`:

| file | contents |
|---|---|
| `PredictorData2024.xlsx` | Goyal-Welch predictors: index, dividends, earnings, rates, spreads |
| `VIX_History.csv` | daily CBOE VIX |
| `nvix_and_categories_timeseries_mar2016.xlsx` | Manela-Moreira news-implied volatility |
| `SENTIMENT.xlsx` | Baker-Wurgler workbook, used for the recession flag |
| `JLMZ_MS_Index2017.xlsx` | published Manager Sentiment Index |

Filing length is separate from these monthly inputs. `sentiment_extraction_results/filing_word_counts.parquet`
holds `N_Words` and `N_Unique_Words` from the Loughran-McDonald 10-X summaries for every
filing in `sent_growth_fin_risk_10k_10q.parquet`. The regressions below do not use it.

In [ ]:
PATHS = {
    "goyal_welch": INPUT_DATA_DIR / "PredictorData2024.xlsx",
    "vix": INPUT_DATA_DIR / "VIX_History.csv",
    "nvix": INPUT_DATA_DIR / "nvix_and_categories_timeseries_mar2016.xlsx",
    "recession": INPUT_DATA_DIR / "SENTIMENT.xlsx",
    "msi": INPUT_DATA_DIR / "JLMZ_MS_Index2017.xlsx",
}
SERIES_PATH = TIME_AGGREGATION_DIR / "manager_sentiment_monthly.parquet"

for name, path in {**PATHS, "manager sentiment": SERIES_PATH}.items():
    print(f"  {'ok ' if path.exists() else 'MISSING'}  {name:18s} {path.name}")

In [ ]:
sentiment = pd.read_parquet(SERIES_PATH)
sentiment[MONTH] = pd.to_datetime(sentiment[MONTH])

# Which smoothing variant feeds the regressions. The raw monthly mean and the
# backward MA use only information up to month t; "_hp" does not (see below).
SMOOTHING_SUFFIX = ""  # "" | "_ma4" | "_hp1s"

MEASURES = [c for c in sentiment.columns
            if c.startswith("polarity_") and not c.endswith(("_ma4", "_hp", "_hp1s"))]
PREDICTORS = {m: f"{m}{SMOOTHING_SUFFIX}" for m in MEASURES}

print("measures  :", MEASURES)
print("predictors:", list(PREDICTORS.values()))
print("span      :", sentiment[MONTH].min().date(), "to", sentiment[MONTH].max().date())

### Sign convention

Taken from the strategy definitions instead of being applied by hand. For
`financial_risk` the "positive" verbalizer words are *low-risk* words
("manageable", "negligible"), so a higher polarity already means *less* perceived
risk, i.e. good news -- the same direction as growth sentiment. No flip needed.

In [ ]:
STRATEGY_ALIASES = {"growth_llama": "growth", "growth_MB": "growth"}

for measure in MEASURES:
    name = measure.removeprefix("polarity_")
    strategy = STRATEGIES.get(STRATEGY_ALIASES.get(name, name))
    if strategy is None:
        print(f"  {measure}: no matching strategy, check the sign yourself")
        continue
    direction = "higher = good news" if strategy.higher_is_good else "higher = bad news"
    print(f"  {measure}: {direction}")
    print(f"      {strategy.description}")

## Market data and horizon returns

In [ ]:
SAMPLE_START = "200212"  # one month of lead-in so the first excess return is defined

gw = md.load_goyal_welch(PATHS["goyal_welch"], start=SAMPLE_START)
market = md.build_excess_market_returns(gw, horizons=HORIZONS, skip_months=0)

CONTROLS = list(md.GOYAL_WELCH_CONTROLS)
print("controls:", CONTROLS)
market[["excess_return", *md.return_columns(HORIZONS)]].describe().round(4)

In [ ]:
# Sanity check on the horizon convention: net_cum_return1 at t must compound
# month t and month t+1.
check = market[[MONTH, "excess_return", "net_cum_return1"]].head(6).copy()
check["next_month_return"] = market["excess_return"].shift(-1)
check["t_and_t+1_compounded"] = (1 + check["excess_return"]) * (1 + check["next_month_return"]) - 1
print(check.round(6).to_string(index=False))

expected = (1 + market["excess_return"]) * (1 + market["excess_return"].shift(-1)) - 1
both = market["net_cum_return1"].notna() & expected.notna()
print("\nnet_cum_return1 == (1+r_t)(1+r_t+1)-1:",
      np.allclose(market.loc[both, "net_cum_return1"], expected[both]))

In [ ]:
msi = md.load_msi(PATHS["msi"])
vix = md.load_vix(PATHS["vix"])
nvix = md.load_nvix(PATHS["nvix"])
recession = md.load_recession(PATHS["recession"])

data = md.merge_monthly(market, sentiment, msi, vix, nvix, recession)
print()
print("merged shape:", data.shape)

In [ ]:
# Standardize predictors and moderators so coefficients are per standard
# deviation and comparable across measures.
TO_STANDARDIZE = [*PREDICTORS.values(), "msi", "vix", "nvix"]

for col in TO_STANDARDIZE:
    if col in data.columns:
        data[f"{col}_z"] = zscore(data[col])

SENTIMENT_Z = {m: f"{p}_z" for m, p in PREDICTORS.items()}
MSI_Z, VIX_Z, NVIX_Z = "msi_z", "vix_z", "nvix_z"

data[[*SENTIMENT_Z.values(), MSI_Z, VIX_Z, NVIX_Z, "recession"]].describe().round(3)

## Descriptives

How the measures relate to each other and to the published index. `n=` in each
cell is the pairwise sample -- the measures rarely cover the same months.

In [ ]:
ec.correlation_with_pvalues(data, [*SENTIMENT_Z.values(), MSI_Z, VIX_Z, NVIX_Z])

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

smoothed = data.set_index(MONTH)
for measure, predictor in SENTIMENT_Z.items():
    ax.plot(smoothed.index, zscore(smoothed[predictor].rolling(4).mean()),
            linewidth=2, label=measure.removeprefix("polarity_"))
ax.plot(smoothed.index, zscore(smoothed[MSI_Z].rolling(4).mean()),
        linewidth=2, linestyle="--", color="black", label="MSI (Jiang et al.)")

# shade each contiguous recession block
in_recession = smoothed["recession"].fillna(0).astype(bool)
blocks = (in_recession != in_recession.shift()).cumsum()[in_recession]
for _, months in smoothed.index.to_series()[in_recession].groupby(blocks):
    ax.axvspan(months.min(), months.max(), color="grey", alpha=0.15)

ax.set_title("Manager sentiment measures, 4-month MA, standardized (shaded: recessions)")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## Predictive regressions

One regression per horizon: `net_cum_return{h} ~ sentiment_z`, Newey-West with
*h* lags. Read `nobs` next to every coefficient -- the long horizons lose the end
of the sample, and the measures do not share a sample.

In [ ]:
# Benchmark: the published Manager Sentiment Index over the same months.
_, msi_table = ec.predictive_regression(data, MSI_Z, HORIZONS)
msi_table

In [ ]:
models = {}
for measure, predictor in SENTIMENT_Z.items():
    models[measure], table = ec.predictive_regression(data, predictor, HORIZONS)
    print(f"\n{measure}  ({predictor})")
    print(table.to_string())

In [ ]:
# Side by side, including the published index.
ec.compare_measures(data, [MSI_Z, *SENTIMENT_Z.values()], HORIZONS)

In [ ]:
# With the Goyal-Welch controls.
for measure, predictor in SENTIMENT_Z.items():
    _, table = ec.predictive_regression(data, predictor, HORIZONS, controls=CONTROLS)
    print(f"\n{measure} + {len(CONTROLS)} controls")
    print(table.to_string())

### Original sample period

January 2003 to December 2014, the window Jiang et al. report, so the numbers
can be put next to the published ones.

In [ ]:
JIANG_PERIOD = ("2003-01-01", "2014-12-31")
jiang = data[data[MONTH].between(*JIANG_PERIOD)].reset_index(drop=True)
print(f"{len(jiang)} month(s) in the original window")

ec.compare_measures(jiang, [MSI_Z, *SENTIMENT_Z.values()], HORIZONS)

## Uncertainty interactions

`return ~ sentiment + uncertainty + sentiment x uncertainty`. The table reports
the interaction coefficient: whether sentiment predicts returns differently when
uncertainty is high.

In [ ]:
MODERATORS = {"VIX": VIX_Z, "NVIX": NVIX_Z, "recession": "recession"}

interaction_models = {}
for measure, predictor in SENTIMENT_Z.items():
    for label, moderator in MODERATORS.items():
        if data[moderator].notna().sum() < 24:
            print(f"skipping {measure} x {label}: only {data[moderator].notna().sum()} month(s)")
            continue
        term = md.add_interaction(data, predictor, moderator)
        fitted, table = ec.interaction_regression(
            data, predictor, moderator, HORIZONS, interaction=term
        )
        interaction_models[(measure, label)] = fitted
        print(f"\n{measure} x {label}  (coefficient on the interaction)")
        print(table.to_string())

In [ ]:
# Marginal effect at the 10th, 50th and 90th percentile of the moderator.
# Percentiles come from the regression's own sample, so the plotted range is the
# range the coefficients were estimated on.
PLOT_HORIZON = 12

pairs = [k for k in interaction_models if k[1] in ("VIX", "NVIX")]
if pairs:
    fig, axes = plt.subplots(1, len(pairs), figsize=(7 * len(pairs), 5), squeeze=False)
    for ax, (measure, label) in zip(axes[0], pairs):
        results = interaction_models[(measure, label)][PLOT_HORIZON]
        predictor = SENTIMENT_Z[measure]
        moderator = MODERATORS[label]
        effects = ec.marginal_effect(results, predictor, moderator, f"{predictor}_x_{moderator}")
        ec.plot_marginal_effect(
            effects, predictor, label,
            ylabel=f"Predicted {PLOT_HORIZON}-month cumulative excess return", ax=ax,
        )
    plt.tight_layout()
    plt.show()

### Growth sentiment moderated by financial risk

The specification from the financial-risk notebook: does the growth signal behave
differently when managers also sound worried about financial risk? Runs only when
both measures are present.

In [ ]:
GROWTH, RISK = "polarity_growth", "polarity_financial_risk"

if GROWTH in SENTIMENT_Z and RISK in SENTIMENT_Z:
    growth_z, risk_z = SENTIMENT_Z[GROWTH], SENTIMENT_Z[RISK]
    term = md.add_interaction(data, growth_z, risk_z)
    fitted, table = ec.interaction_regression(data, growth_z, risk_z, HORIZONS, interaction=term)
    print("growth x financial risk (coefficient on the interaction)")
    print(table.to_string())
    print()
    print(fitted[PLOT_HORIZON].summary())
else:
    print(f"need both {GROWTH} and {RISK}; have {list(SENTIMENT_Z)}")

In [ ]:
if GROWTH in SENTIMENT_Z and RISK in SENTIMENT_Z:
    effects = ec.marginal_effect(
        fitted[PLOT_HORIZON], SENTIMENT_Z[GROWTH], SENTIMENT_Z[RISK],
        f"{SENTIMENT_Z[GROWTH]}_x_{SENTIMENT_Z[RISK]}",
    )
    ec.plot_marginal_effect(
        effects, SENTIMENT_Z[GROWTH], "financial risk polarity",
        ylabel=f"Predicted {PLOT_HORIZON}-month cumulative excess return",
    )
    plt.tight_layout()
    plt.show()

## Robustness: what the conventions are worth

The two choices flagged at the top, re-run with the alternative settings so the
difference is visible rather than asserted: a window that starts at *t+1* instead
of *t*, and a fixed 12-lag HAC covariance at every horizon.

In [ ]:
measure = next(iter(SENTIMENT_Z))
predictor = SENTIMENT_Z[measure]

_, baseline = ec.predictive_regression(data, predictor, HORIZONS)
_, fixed_lags = ec.predictive_regression(data, predictor, HORIZONS, maxlags=12)

forward = md.merge_monthly(
    md.build_excess_market_returns(gw, horizons=HORIZONS, skip_months=1),
    sentiment,
)
forward[predictor] = zscore(forward[PREDICTORS[measure]])
_, forward_only = ec.predictive_regression(forward, predictor, HORIZONS)

comparison = pd.concat(
    {
        "t..t+h, h lags (default)": baseline[["coef", "t", "r2_adj", "nobs"]],
        "t..t+h, 12 lags": fixed_lags[["coef", "t", "r2_adj", "nobs"]],
        "t+1..t+h, h lags": forward_only[["coef", "t", "r2_adj", "nobs"]],
    },
    axis=1,
)
print(f"{measure}: effect of the two conventions")
comparison